# 🦜 BirdCLEF 2026 - Target-Domain Pseudo-Labeling

> **Objective:** Overcome the CV-LB Domain Shift via the Multi-Iterative Noisy Student paradigm.
>
> **Hardware:** Kaggle 2x NVIDIA Tesla T4.
>
> **Prerequisites:** Attach the output of [Training Notebook](https://www.kaggle.com/code/emanuellcs/birdclef-2026-training) containing all 5 exported folds (`birdclef2026_foldX.onnx`) and of [I/O Preprocessing Notebook](https://www.kaggle.com/code/emanuellcs/birdclef-2026-i-o-preprocessing) (`label_map.csv`).

---

## The "Noisy Student" Power Transform & Ensemble Distillation

Our Stage 1 models were trained on clean, focal Xeno-Canto recordings. Directly using a single model's predictions on noisy target-domain soundscapes as ground truth causes "confirmation bias"—it confidently learns its own mistakes.

To safely distill knowledge from the target domain, this notebook implements a robust, multi-GPU ensemble pipeline:

* **K-Fold Ensembling:** We run parallel inference across 5 structurally diverse fold checkpoints, averaging their logits to suppress single-model blind spots.
* **Temporal Smoothing:** A 1D moving average is applied across consecutive overlapping windows to bridge "flickering" predictions and replicate realistic vocalization lengths.
* **The Babych Power Transform (1st Place, BirdCLEF 2025):** We apply an exponent $\gamma = 2.0$ to the ensembled probability distribution:

$$P_{transformed} = P_{raw}^\gamma$$

* **Dynamic Class-Specific Thresholding:** Instead of a uniform confidence cutoff, we dynamically calculate the 95th percentile threshold for *each specific class*, ensuring rare species are not systematically deleted from the pseudo-labels.

By setting $\gamma = 2.0$, we mathematically crush low-confidence background noise toward `0.0` while preserving high-confidence avian detections near `1.0`. These sharpened, soft-target distributions will be saved as a CSV and injected into our `RAMBirdDataset` during Stage 2 training via Additive MixUp.

In [ ]:
!pip install -q onnxruntime-gpu

In [ ]:
%%writefile /kaggle/working/pseudo_worker.py
import os
import numpy as np
import pandas as pd
import soundfile as sf
import onnxruntime as ort
from scipy.ndimage import uniform_filter1d
from typing import List, Tuple
from pathlib import Path
import warnings
from multiprocessing import Queue, shared_memory

warnings.filterwarnings('ignore')

# =============================================================================
# 1. AUDIO PROCESSING PIPELINE
# =============================================================================
def extract_overlapping_windows(wav: np.ndarray, cfg: dict) -> Tuple[np.ndarray, np.ndarray]:
    w_samp, h_samp = cfg['window_samples'], cfg['hop_samples']
    if len(wav) < w_samp:
        wav = np.pad(wav, (0, w_samp - len(wav)))
    elif (len(wav) - w_samp) % h_samp != 0:
        pad_len = h_samp - ((len(wav) - w_samp) % h_samp)
        wav = np.pad(wav, (0, pad_len))
        
    num_chunks = (len(wav) - w_samp) // h_samp + 1
    shape = (num_chunks, w_samp)
    strides = (wav.strides[0] * h_samp, wav.strides[0])
    batches = np.lib.stride_tricks.as_strided(wav, shape=shape, strides=strides)
    
    timestamps = np.zeros((num_chunks, 2), dtype=np.float32)
    for i in range(num_chunks):
        timestamps[i, 0] = (i * h_samp) / cfg['sample_rate']
        timestamps[i, 1] = timestamps[i, 0] + cfg['window_sec']
        
    # .copy() ensures the array is contiguous in memory before it goes to SharedMemory
    return batches.copy(), timestamps

# =============================================================================
# 2. THE PRODUCER (Strictly handles Disk I/O & Decoding)
# =============================================================================
def producer_worker(file_list: List[Path], cfg: dict, task_queue: Queue):
    """Decodes audio and writes directly to Shared Memory."""
    for filepath in file_list:
        try:
            wav, sr = sf.read(filepath, dtype='float32')
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
                
            chunks, timestamps = extract_overlapping_windows(wav, cfg)
            
            # 1. Allocate block in RAM
            shm = shared_memory.SharedMemory(create=True, size=chunks.nbytes)
            # 2. Create NumPy view of that RAM
            shared_array = np.ndarray(chunks.shape, dtype=chunks.dtype, buffer=shm.buf)
            # 3. Copy audio data into RAM
            shared_array[:] = chunks[:] 
            
            # 4. Pass only the metadata strings (Lightning fast IPC, no pickling!)
            task_queue.put((shm.name, chunks.shape, chunks.dtype, filepath.name, timestamps))
            
            # Producer closes its handle (Consumer is responsible for deletion)
            shm.close()
            
        except Exception:
            continue # Silently skip corrupted audio
            
    # Send Poison Pills to signal the 2 GPU Consumers to shut down
    task_queue.put(None)
    task_queue.put(None)

# =============================================================================
# 3. THE CONSUMERS (Strictly handles GPU Inference)
# =============================================================================
def consumer_worker(gpu_id: int, cfg: dict, classes: List[str], task_queue: Queue, result_queue: Queue):
    """Pulls addresses from queue, reads Shared Memory, runs ONNX."""
    sessions = []
    for model_path in cfg['onnx_paths']:
        if not os.path.exists(model_path): continue
        providers = [
            ('CUDAExecutionProvider', {
                'device_id': gpu_id,
                'cudnn_conv_algo_search': 'EXHAUSTIVE',
                'arena_extend_strategy': 'kNextPowerOfTwo',
            }),
            'CPUExecutionProvider'
        ]
        sess_opts = ort.SessionOptions()
        sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        sess_opts.intra_op_num_threads = 1
        sess_opts.inter_op_num_threads = 1
        sessions.append(ort.InferenceSession(model_path, sess_options=sess_opts, providers=providers))
        
    if not sessions:
        result_queue.put("ERROR")
        return

    while True:
        item = task_queue.get()
        if item is None:
            break # Poison pill received
            
        shm_name, shape, dtype, filename, timestamps = item
        
        # 1. Attach to the exact memory block the Producer created
        shm = shared_memory.SharedMemory(name=shm_name)
        chunks = np.ndarray(shape, dtype=dtype, buffer=shm.buf)
        
        ensemble_logits = np.zeros((len(chunks), len(classes)), dtype=np.float32)
        
        # 2. Micro-batch inference
        for i in range(0, len(chunks), cfg['batch_size']):
            batch = chunks[i : i + cfg['batch_size']]
            batch_logits = np.zeros((len(batch), len(classes)), dtype=np.float32)
            
            for sess in sessions:
                batch_logits += sess.run(None, {'waveform': batch})[0]
            batch_logits /= len(sessions)
            
            ensemble_logits[i : i + cfg['batch_size']] = batch_logits
            
        # 3. Free the Shared Memory immediately so the Producer can allocate more
        shm.close()
        shm.unlink()
        
        # 4. Post-processing
        raw_probs = 1.0 / (1.0 + np.exp(-ensemble_logits))
        smoothed_probs = uniform_filter1d(raw_probs, size=3, axis=0, mode='nearest')
        transformed_probs = np.power(smoothed_probs, cfg['gamma'])
        
        df = pd.DataFrame(transformed_probs, columns=classes)
        df.insert(0, 'filename', filename)
        df.insert(1, 'start_time', timestamps[:, 0])
        df.insert(2, 'end_time', timestamps[:, 1])
        
        # Send completed DataFrame back to Orchestrator
        result_queue.put(df)
        
    result_queue.put("DONE")

In [ ]:
import sys
import shutil
import multiprocessing
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Add the Kaggle working directory to the path so Python can find pseudo_worker.py
sys.path.insert(0, '/kaggle/working')
from pseudo_worker import producer_worker, consumer_worker

# =============================================================================
# REUNITE ONNX MODELS WITH THEIR EXTERNAL DATA FILES
# =============================================================================
input_dir = Path("/kaggle/input/notebooks/emanuellcs/birdclef-2026-training") 
local_model_dir = Path("/kaggle/working/models")
local_model_dir.mkdir(exist_ok=True)

valid_onnx_paths = []
print("[SETUP] Staging RAW ONNX models for GPU execution...")

for fold in range(5):
    raw_onnx = input_dir / f"fold_{fold}" / f"birdclef2026_fold{fold}_raw.onnx"
    raw_data = input_dir / f"fold_{fold}" / f"birdclef2026_fold{fold}_raw.onnx.data"
    
    local_raw_onnx = local_model_dir / f"birdclef2026_fold{fold}_raw.onnx"
    local_raw_data = local_model_dir / f"birdclef2026_fold{fold}_raw.onnx.data"
    
    if raw_onnx.exists() and raw_data.exists():
        shutil.copy(raw_onnx, local_raw_onnx)
        shutil.copy(raw_data, local_raw_data)
        valid_onnx_paths.append(str(local_raw_onnx))
        print(f"  ✅ Fold {fold} raw model staged for CUDA.")
        
if not valid_onnx_paths:
    raise FileNotFoundError("CRITICAL ERROR: Could not find the RAW ONNX files.")

# =============================================================================
# GLOBAL CONFIGURATION
# =============================================================================
CFG = dict(
    unlabelled_dir = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes"),
    onnx_paths     = valid_onnx_paths,  
    label_map_path = "/kaggle/input/notebooks/emanuellcs/birdclef-2026-i-o-preprocessing/label_map.csv",
    output_csv     = "/kaggle/working/pseudo_labels_gamma1.csv",
    
    sample_rate    = 32000,
    window_sec     = 5.0,
    hop_sec        = 2.5,  
    gamma          = 1.0,  
    batch_size     = 256,   
    dynamic_threshold_percentile = 0.70
)

CFG['window_samples'] = int(CFG['window_sec'] * CFG['sample_rate'])
CFG['hop_samples']    = int(CFG['hop_sec'] * CFG['sample_rate'])

# =============================================================================
# MAIN ORCHESTRATOR
# =============================================================================
def run_distributed_pseudo_labeling(cfg: dict):
    label_map = pd.read_csv(cfg['label_map_path']).sort_values('label_id')
    classes = label_map['primary_label'].tolist()
    
    audio_files = sorted(list(cfg['unlabelled_dir'].rglob('*.ogg')))
    print(f"\n[DATA] Found {len(audio_files)} soundscapes. Booting SharedMemory Pipeline...")
    
    ctx = multiprocessing.get_context('spawn')
    
    # maxsize=20 acts as a throttle to prevent RAM exhaustion
    task_queue = ctx.Queue(maxsize=20)
    result_queue = ctx.Queue()
    
    # 1. Start the Producer (Core 0)
    producer = ctx.Process(target=producer_worker, args=(audio_files, cfg, task_queue))
    producer.start()
    
    # 2. Start the Consumers (Core 1 & 2 -> GPU 0 & 1)
    consumers = []
    for gpu_id in range(2):
        p = ctx.Process(target=consumer_worker, args=(gpu_id, cfg, classes, task_queue, result_queue))
        p.start()
        consumers.append(p)
        
    # 3. Collect Results cleanly in the main thread
    all_dfs = []
    finished_consumers = 0
    
    with tqdm(total=len(audio_files), desc="Inference Progress", unit="file") as pbar:
        while finished_consumers < 2:
            res = result_queue.get()
            if isinstance(res, str) and res == "DONE":
                finished_consumers += 1
            elif isinstance(res, str) and res == "ERROR":
                raise RuntimeError("A GPU worker failed to initialize ONNX models.")
            else:
                all_dfs.append(res)
                pbar.update(1)
                
    # 4. Cleanup Processes
    producer.join()
    for p in consumers:
        p.join()
        
    print("\n[MERGE] Aggregating results from GPUs...")
    final_df = pd.concat(all_dfs, ignore_index=True)
    
    print(f"[FILTER] Applying {cfg['dynamic_threshold_percentile']*100}% dynamic thresholding...")
    class_thresholds = final_df[classes].quantile(cfg['dynamic_threshold_percentile'])
    keep_mask = (final_df[classes] >= class_thresholds).any(axis=1)
    filtered_df = final_df[keep_mask].copy()
    
    filtered_df.to_csv(cfg['output_csv'], index=False)
    
    print("\n" + "="*50)
    print("✅ Pipeline Complete!")
    print(f"Original Rows    : {len(final_df):,}")
    print(f"Filtered Rows    : {len(filtered_df):,}")
    print(f"Output saved to  : {cfg['output_csv']}")
    print("="*50)

if __name__ == '__main__':
    run_distributed_pseudo_labeling(CFG)